# Base Clients

> The core abstraction for different FL Clients. Any gneral client functionality resides here.

In [ ]:
#| default_exp clients.base_client

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import math

from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from fedai.metrics import Metrics


<torch._C.Generator>

## BaseClient

This class implements the basic abstraction of a client and is considered the base class for all type of clients

In [ ]:
#| export
class BaseClient:
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        self.id = id 
        self.cfg = cfg 
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.t = t
    
        self.train_loader = train_loader
        self.test_loader = test_loader
        
        for key, value in state.items():
            setattr(self, key, value) 

        self.criterion = criterion

        self.training_metrics = Metrics(list(self.cfg.training_metrics))  
        self.test_metrics = Metrics(list(self.cfg.test_metrics))  

        self.data_key = 'image'#train_loader.dataset.x_key
        self.label_key = 'label'#train_loader.dataset.y_key

### Helpers

We will adjust the string reprsntation of the client abstraction to make it more meaningful.

In [ ]:
#| export
@patch
def __str__(self: BaseClient) -> str:
    return f'''{self.__class__.__name__}: {self.__class__.__name__}
    Index : {self.id}
    Model: {self.model.__class__.__name__}
    Criterion: {self.criterion.__class__.__name__}
    Optimizer: {self.optimizer.__class__.__name__}'''


In [ ]:
#| export
@patch
def send_to_device(self: BaseClient, batch):
    return {k: v.to(self.device) for k, v in batch.items()}

### Training

In [ ]:
#| export
@patch
def fit(self: BaseClient):
    
    self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            outputs = self.model(X)
            
            loss = self.criterion(outputs, y)
            loss.backward()
            self.optimizer.step()


### Local Client Evaluation




In [ ]:
#| export
@patch
def train_test_stats(self: BaseClient, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch[self.data_key], batch[self.label_key]
    logits = self.model(X)
    loss = self.criterion(logits, y)
    y_pred = logits.argmax(dim=-1)
    y_true = batch[self.label_key]

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: BaseClient, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.model = self.model.to(self.device)
    self.model.eval()
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    return {"loss": avg_loss, "metrics": total_metrics}


### Checkpointing Client State


In [ ]:
#| export
@patch
def save_state(self: BaseClient, save_to_disk= False):  
    # save the model to self.cfg.save_dir/comm_round/f"local_output_{id}"/state.pth

    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()

    if save_to_disk:
        state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()